In [111]:
from pathlib import Path

print(Path.cwd())

c:\Users\ameli\Documents\AI Datasets\pionex-cleaning-project


# Environment Setup
### importing the necessary libraries

In [112]:
import pandas as pd
import numpy as np
import logging
from pathlib import Path
from datetime import datetime

### 2. Folder Structure Setup

This section creates all folders required for the cleaning workflow.

In [113]:
from pathlib import Path

RAW_FOLDER = Path("records/raw")
PROCESSED_FOLDER = Path("records/processed")
GARBAGE_FOLDER = Path("records/garbage")
INGEST_FOLDER = Path("records/to-be-ingested")
LOG_FOLDER = Path("logs")

folders = [
    RAW_FOLDER,
    PROCESSED_FOLDER,
    GARBAGE_FOLDER,
    INGEST_FOLDER,
    LOG_FOLDER
]

for folder in folders:
    folder.mkdir(parents=True, exist_ok=True)

print("Folder structure created successfully.")

Folder structure created successfully.


### 3. Logging Setup

This section configures logging to track the execution of the cleaning workflow.

In [114]:
import logging

logging.basicConfig(
    filename="logs/pionex_cleaning.log",
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s"
)

logging.info("Pipeline started.")

print("Logging configured successfully.")

Logging configured successfully.


### Just verifying the log file was created

In [115]:
from pathlib import Path

print(Path("logs/pionex_cleaning.log").exists())

True


### Verifying the correct file name 

In [116]:
from pathlib import Path

for file in Path("records/raw").iterdir():
    print(file.name)

pionex_raw.xlsx


# The following codes will be used to generate the Pre-Analysis info.

### This code highlights the information in the first couple entries the dataset

In [117]:
df_raw = pd.read_excel("records/raw/pionex_raw.xlsx")

print(df_raw.head())

                          Email   First Name   Last Name        Phone  \
0          lgreaves35@gmail.com         Luke     Greaves  61431304164   
1    jeanbaptist.tama@gmail.com  Jeanbaptist        Tama   6785555070   
2  christianantondean@gmail.com    Christian        Dean  12424249200   
3       manuelefereti@gmail.com      Manuele        kuki  61415050847   
4       chrissi.mac48@gmail.com  Christopher  mcguinness  61448469538   

           Country Lang  RegistrationDate   BrandCode  
0        Australia   EN  18.08.2021 23:57  pionex.com  
1  Solomon Islands   EN  18.08.2021 23:57  pionex.com  
2          Bahamas   EN  18.08.2021 23:54  pionex.com  
3        Australia   EN  18.08.2021 23:54  pionex.com  
4        Australia   EN  18.08.2021 23:54  pionex.com  


### This code highlights the entries at the end of the dataset

In [118]:
print(df_raw.tail())

                            Email         First Name   Last Name        Phone  \
241928         nor80ola@yahoo.com  Ola Erik Fossland         NaN   4797501233   
241929  mr_kristensen@hotmail.com       Svein Jorgen  Kristensen   4791197877   
241930       jclimbourg@gmail.com           Zoumrati     Ousseni  33610011334   
241931        olebredal@gmail.com         ole bredal         NaN   4751893131   
241932         chrhol@carnegie.no          Christian       Holme   4793409355   

       Country Lang RegistrationDate   BrandCode  
241928  Norway   EN  18.08.2021 4:54  pionex.com  
241929  Norway   EN  18.08.2021 4:51  pionex.com  
241930  France   EN  18.08.2021 4:51  pionex.com  
241931  Norway   EN  18.08.2021 4:51  pionex.com  
241932  Norway   EN  18.08.2021 4:51  pionex.com  


### 5. Dataset Size Assessment

The dataset contains approximately 242,000 records, therefore the current dataset is small enogh so it can be successfully loaded into memory.

In [119]:
print(df_raw.shape)

(241933, 8)


### . Highlights missing values

In [120]:
print(df_raw.isnull().sum())

Email                   1
First Name             20
Last Name           25612
Phone                   0
Country                 0
Lang                    0
RegistrationDate     1792
BrandCode               0
dtype: int64


## Missing Value Analysis

The dataset contains missing values in several fields. The most significant issues were identified in:

- Last Name
- RegistrationDate

These fields will be evaluated during the cleaning phase.

### This code highlights the duplicated entries

In [121]:
print(df_raw.duplicated().sum())

0


In [122]:
duplicate_count = df_raw.duplicated().sum()

print(f"Duplicate Rows: {duplicate_count}")

Duplicate Rows: 0


In [123]:
df_raw[df_raw["RegistrationDate"].isnull()].head()

,Email,First Name,Last Name,Phone,Country,Lang,RegistrationDate,BrandCode
6997,neilburton00@yahoo.com,burton,NaN,448442000000,United Kingdom,EN,NaN,pionex.com
7010,noahgcriddle@gmail.com,Noah,Criddle,13065318476,Canada,EN,NaN,pionex.com
7028,skaspars@gmail.com,Kaspars,NaN,37127727651,Latvia,EN,NaN,pionex.com
7032,maronifd@gmail.com,Lani,Ki,64273243408,New Zealand,EN,NaN,pionex.com
7033,alisonh897@gmail.com,Alison,Hamilton,64221662067,New Zealand,EN,NaN,pionex.com


### This code highlights the various datatypes in the dataset

In [124]:
print(df_raw.dtypes)

Email               object
First Name          object
Last Name           object
Phone               object
Country             object
Lang                object
RegistrationDate    object
BrandCode           object
dtype: object


### This code highlights the various columns in the dataset

In [125]:
print(df_raw.columns.tolist())

['Email', 'First Name', 'Last Name', 'Phone', 'Country', 'Lang', 'RegistrationDate', 'BrandCode']


### Here I've created a Summary Table based on the information in the dataset

In [126]:
pre_analysis = pd.DataFrame({
    "Data Type": df_raw.dtypes,
    "Missing Values": df_raw.isnull().sum()
})

pre_analysis

,Data Type,Missing Values
Email,object,1
First Name,object,20
Last Name,object,25612
Phone,object,0
Country,object,0
Lang,object,0
RegistrationDate,object,1792
BrandCode,object,0


# Logging the Analysis of the dataset

In [127]:
logging.info("Pre-analysis completed")

In [128]:
df_raw.duplicated().sum()
df_raw.isnull().sum()
df_raw["Lang"].value_counts()
df_raw["BrandCode"].value_counts()

BrandCode
pionex.com    241933
Name: count, dtype: int64

# 4. Dataset Loading

This section loads the raw Pionex dataset and records the initial row count.

In [129]:
raw_file = "records/raw/pionex_raw.xlsx"

df_raw = pd.read_excel(raw_file)

original_rows = len(df_raw)

print(f"Rows loaded: {original_rows}")

logging.info(f"Loaded dataset with {original_rows} rows")

Rows loaded: 241933


# 6. Cleaning Functions

In [130]:
duplicate_rows = df_raw[df_raw.duplicated()]
duplicate_rows.to_csv(
    "records/garbage/pionex_duplicates.csv",
    index=False
)
logging.info(
    f"Duplicate records saved: {len(duplicate_rows)}"
)

# The following steps were used to clean the dataset

### This code will be used to standardize the column names

In [131]:
import re

def standardize_columns(df):
    df.columns = [
        re.sub(r'(?<!^)(?=[A-Z])', '_', col)
        .strip()
        .lower()
        .replace(" ", "_")
        for col in df.columns
    ]
    return df

In [132]:
df_clean = standardize_columns(df_raw)

print(df_clean.columns.tolist())

['email', 'first__name', 'last__name', 'phone', 'country', 'lang', 'registration_date', 'brand_code']


### This code will remove the duplicate entries

In [133]:
def remove_duplicates(df):
    return df.drop_duplicates()

# Column Removal
 This code block will remove unwanted fields from the datase. I chose to drop the language, registration date and brand code columns because they were not required for the intended downstream, so as to maintain traceability the removed records were exported to the garbage folder.
 To maintain transparency and support auditing, these columns were not immediately discarded. Instead, they were first extracted and exported to a garbage file located in the records/garbage folder. This ensures that the original information remains available for future review if required.

After exporting the removed data, the columns were dropped from the working dataset to simplify the structure and reduce unnecessary attributes. The resulting dataset contains only the fields required for subsequent cleaning, validation, and ingestion processes.

In [134]:
def remove_unneeded_columns(df):
    
    removed_columns = df[
        ["lang", "registrationdate", "brandcode"]
    ].copy()

    removed_columns.to_csv(
        "records/garbage/pionex_removed_columns.csv",
        index=False
    )

    logging.info(
        "Removed columns saved to garbage file."
    )

    return df.drop(
        columns=[
            "lang",
            "registrationdate",
            "brandcode"
        ]
    )

In [135]:
df_clean = remove_unneeded_columns(df_clean)

KeyError: "['registrationdate', 'brandcode'] not in index"

### First saved version

In [ ]:
df_clean.to_csv(
    "records/processed/pionex_v1.csv",
    index=False
)

logging.info("Version 1 saved.")

### This shows the missing values after the removal of the columns

In [ ]:
df_clean.isnull().sum()

email             1
first_name       20
last_name     25612
phone             0
country           0
dtype: int64

### Because there are so many missing values for the last name field especially I chose not to tamper with that information and leave it as is. However i did remove the missing email and first name because it was have had a negligible impact on the overall dataset.

In [ ]:
df_clean = df_clean.dropna(
    subset=["email"]
)

In [ ]:
df_clean = df_clean.dropna(
    subset=["first_name"]
)

# Post Analysis 
This table shows the basic information now that the dataset has been cleaned.

In [ ]:
print(df_clean.shape)

print(df_clean.isnull().sum())

print(df_clean.duplicated().sum())

(241912, 5)
email             0
first_name        0
last_name     25598
phone             0
country           0
dtype: int64
0


### Saving second version

In [ ]:
df_clean.to_csv(
    "records/processed/pionex_v2.csv",
    index=False
)

In [ ]:
print(df_clean.isnull().sum())

email             0
first_name        0
last_name     25598
phone             0
country           0
dtype: int64


### Row Reconciliation
this is a comparisionn of the information before and after the dataset was cleanedpresented in tabular form

In [ ]:
original_rows = len(df_raw)

removed_columns_rows = 0

duplicate_rows_count = len(duplicate_rows)


final_rows = len(df_clean)

In [ ]:
print("Original rows:", len(df_raw))
print("Current rows:", len(df_clean))
print("Duplicates:", df_clean.duplicated().sum())
print("\nMissing values:")
print(df_clean.isnull().sum())

Original rows: 241933
Current rows: 241933
Duplicates: 0

Missing values:
email             1
first_name       20
last_name     25612
phone             0
country           0
dtype: int64


In [ ]:
original_rows = len(df_raw)

missing_email_count = 1

final_rows = len(df_clean)

In [ ]:
reconciliation = pd.DataFrame({
    "Metric": [
        "Original Rows",
        "Missing Email Removed",
        "Final Rows"
    ],
    "Count": [
        original_rows,
        missing_email_count,
        final_rows
    ]
})

reconciliation

,Metric,Count
0,Original Rows,241933
1,Missing Email Removed,1
2,Final Rows,241912


In [ ]:
final_rows = len(df_clean)

print(final_rows)

241912


In [ ]:
missing_email_count = 1
missing_first_name_count = 20

rows_removed = (
    missing_email_count +
    missing_first_name_count
)

print(f"Rows removed: {rows_removed}")
print(f"Original rows: {original_rows}")
print(f"Final rows: {final_rows}")

Rows removed: 21
Original rows: 241933
Final rows: 241912


In [ ]:
assert original_rows == rows_removed + final_rows

#POST ANALYSIS


Following the cleaning process, all records containing missing email addresses and missing first names were removed from the dataset. Duplicate analysis confirmed that no duplicate records were present.

The `last_name` field continues to contain missing values. These records were intentionally retained because removing them would have resulted in a significant loss of data. Since other identifying information such as email address, phone number, and country remained available, the records were considered suitable for retention.

The final dataset contains no missing values in the email, first_name, phone, or country fields and contains no duplicate records.


In [ ]:
print("Final Dataset Shape:")
print(df_clean.shape)

print("\nRemaining Missing Values:")
print(df_clean.isnull().sum())

print("\nRemaining Duplicates:")
print(df_clean.duplicated().sum())

Final Dataset Shape:
(241912, 5)

Remaining Missing Values:
email             0
first_name        0
last_name     25598
phone             0
country           0
dtype: int64

Remaining Duplicates:
0


In [ ]:
reconciliation = pd.DataFrame({
    "Metric": [
        "Original Rows",
        "Missing Email Removed",
        "Missing First Name Removed",
        "Final Rows"
    ],
    "Count": [
        original_rows,
        missing_email_count,
        missing_first_name_count,
        final_rows
    ]
})

reconciliation

,Metric,Count
0,Original Rows,241933
1,Missing Email Removed,1
2,Missing First Name Removed,20
3,Final Rows,241912


In [ ]:
assert original_rows == (
    missing_email_count
    + missing_first_name_count
    + final_rows
)

print("Row reconciliation passed.")

Row reconciliation passed.


In [ ]:
print(df_clean.isnull().sum())
print(df_clean.duplicated().sum())

email             0
first_name        0
last_name     25598
phone             0
country           0
dtype: int64
0


1# Saved Version 3 and uploaded it to the to-be-ingested folder.

In [ ]:
df_clean.to_csv(
    "records/processed/pionex_v3.csv",
    index=False
)

In [ ]:
df_clean.to_csv(
    "records/to-be-ingested/pionex_cleaned.csv",
    index=False
)
logging.info("Final cleaned dataset exported.")

In [ ]:
print("FINAL VALIDATION")
print("-" * 40)

print(f"Original Rows: {len(df_raw):,}")
print(f"Final Rows: {len(df_clean):,}")

print("\nMissing Values:")
print(df_clean.isnull().sum())

print("\nDuplicate Rows:")
print(df_clean.duplicated().sum())

print("\nPipeline completed successfully.")

FINAL VALIDATION
----------------------------------------
Original Rows: 241,933
Final Rows: 241,912

Missing Values:
email             0
first_name        0
last_name     25598
phone             0
country           0
dtype: int64

Duplicate Rows:
0

Pipeline completed successfully.


In [ ]:
missing_last_name.to_csv(
    "records/garbage/pionex_missing_last_names.csv",
    index=False
)

# Conclusion

The Pionex dataset cleaning workflow was executed successfully from start to finish with minimal manual intervention.

The workflow included:

* Programmatic folder creation
* Logging implementation
* Dataset loading
* Pre-cleaning analysis
* Column standardization
* Removal of unnecessary columns
* Garbage file generation
* Missing value handling
* Row reconciliation
* Post-cleaning validation
* Final export preparation

A total of 21 records were removed due to missing critical fields. No duplicate records were identified within the dataset. Records with missing last names were retained to prevent unnecessary data loss and were documented through garbage file generation.

The final cleaned dataset was exported to the `to-be-ingested` folder and is ready for downstream ingestion.
